# 提交因子:`f019_pana_h5`(run002 head5)

> ⚠️ 平台只执行下方 code cell 里的 `main()`;定义函数不产生输出,直接上传即可。
> 随提交上传权重附件 **f019_weights.json**(推理必需)。

池感知正交多头神经因子(推理版):训练离线完成(2019-2023 写死窗;残差 IC + 公开库/自有锚(含在榜 f012)相关惩罚 + 级联 Gram-Schmidt 正交多头 + GroupDRO 逐年鲁棒),本 main 仅特征 SQL + 前向推理。训练/筛选全档案:`ai_gen/f012_pana_nn/`。

In [ ]:

def main(datasources, start_date, end_date):
    """f019_pana_h5 —— 池感知正交多头神经因子(AI 子赛道 · 推理版,run002 head5)。

    训练已离线完成(训练窗写死 2019-2023,GroupDRO 逐年鲁棒 + 池相关惩罚 + 级联正交多头;
    详见 ai_gen/f012_pana_nn/),本函数仅推理:载入 f019_weights.json → 平台传入区间样本外预测。
    因子值 = 模型头 5 对 [T=20 日 × 37 个日频微结构原语] 面板的输出,按日截面 zscore。
    """
    import gc
    import json
    import math
    import os

    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import dai

    MIN_BARS = 60
    _SQRT12 = 3.4641016
    T = 20
    HEAD = 5
    PRIM_COLS = [
        "pv_ret_oc", "pv_ret_am", "pv_ret_pm", "pv_ret_first30", "pv_ret_last30",
        "pv_pos_close", "pv_vwap_dev", "pv_vwap_dev_last30", "pv_range", "rd_std",
        "rd_skew", "rd_kurt", "rd_upshare", "rd_trend_eff", "vt_hhi",
        "vt_first30_share", "vt_last30_share", "vt_ret_vol_corr", "vt_absret_vol_corr", "vt_avg_trade_size",
        "vt_amihud", "vt_vw_ret", "ob_ofi1", "ob_ofi5", "ob_dimb_mean",
        "ob_dimb_std", "ob_dimb_last30", "ob_dimb1_mean", "ob_oimb_mean", "ob_ordsize_imb",
        "ob_spread_mean", "ob_spread_std", "ob_spread_last30", "ob_mpd_mean", "ob_slope_asym",
        "ob_quote_int", "ob_book_to_flow",
    ]

    # ==================== 特征层(与训练数据同源:f006 step1 原文) ====================
    def month_chunks(d0, d1):
        """[d0,d1] 切自然月分块,返回 [(start,end)] 字符串日期对。"""
        s, e = pd.Timestamp(d0), pd.Timestamp(d1)
        out, cur = [], s
        while cur <= e:
            me = min(cur + pd.offsets.MonthEnd(0), e)
            if me < cur:
                me = min(cur + pd.offsets.MonthEnd(1), e)
            out.append((cur.strftime("%Y-%m-%d"), me.strftime("%Y-%m-%d")))
            cur = me + pd.Timedelta(days=1)
        return out


    def primitive_sql(bar_table, inst_table, d0, d1):
        """单个月度分块的原语充分统计量 SQL:分钟表 → (instrument, d) 一行聚合。"""
        return f"""
        WITH pool AS (
            SELECT DISTINCT date::DATE AS pd, instrument
            FROM {inst_table}
            WHERE date::DATE BETWEEN DATE '{d0}' AND DATE '{d1}'
        ),
        base AS (
            SELECT t.instrument, t.date, t.date::DATE AS d,
                   t.open, t.high, t.low, t.close,
                   t.volume::DOUBLE AS volume, t.amount::DOUBLE AS amount,
                   t.deal_number::DOUBLE AS deal_number,
                   t.ask_price1 AS ap1, t.bid_price1 AS bp1,
                   t.ask_price5 AS ap5, t.bid_price5 AS bp5,
                   t.ask_volume1::DOUBLE AS av1, t.bid_volume1::DOUBLE AS bv1,
                   -- int32 逐列先 CAST 再相加(2026-07-14 f009 黑盒报错嫌疑点:先加后 CAST 在隐藏窗
                   -- 遇涨跌停巨量挂单会 INT32 OutOfRange;≤2024 数据未触发属侥幸)
                   (t.bid_volume1::DOUBLE + t.bid_volume2::DOUBLE + t.bid_volume3::DOUBLE
                    + t.bid_volume4::DOUBLE + t.bid_volume5::DOUBLE) AS bidv5,
                   (t.ask_volume1::DOUBLE + t.ask_volume2::DOUBLE + t.ask_volume3::DOUBLE
                    + t.ask_volume4::DOUBLE + t.ask_volume5::DOUBLE) AS askv5,
                   (t.bid_num_orders1::DOUBLE + t.bid_num_orders2::DOUBLE + t.bid_num_orders3::DOUBLE
                    + t.bid_num_orders4::DOUBLE + t.bid_num_orders5::DOUBLE) AS bidn5,
                   (t.ask_num_orders1::DOUBLE + t.ask_num_orders2::DOUBLE + t.ask_num_orders3::DOUBLE
                    + t.ask_num_orders4::DOUBLE + t.ask_num_orders5::DOUBLE) AS askn5
            FROM {bar_table} t
            INNER JOIN pool p ON p.instrument = t.instrument AND p.pd = t.date::DATE
            WHERE t.date::DATE BETWEEN DATE '{d0}' AND DATE '{d1}'
        ),
        w AS (
            SELECT *,
                   LAG(close)       OVER win AS close_p,
                   LAG(volume)      OVER win AS volume_p,
                   LAG(amount)      OVER win AS amount_p,
                   LAG(deal_number) OVER win AS deal_p,
                   LAG(bp1)         OVER win AS bp1_p,
                   LAG(ap1)         OVER win AS ap1_p,
                   LAG(bv1)         OVER win AS bv1_p,
                   LAG(av1)         OVER win AS av1_p,
                   LAG(bidv5)       OVER win AS bidv5_p,
                   LAG(askv5)       OVER win AS askv5_p,
                   ROW_NUMBER()     OVER win AS rn,
                   COUNT(*)         OVER (PARTITION BY instrument, d) AS nb
            FROM base
            WINDOW win AS (PARTITION BY instrument, d ORDER BY date)
        ),
        m AS (
            SELECT instrument, d, rn, nb, (nb - rn + 1) AS rn_desc,
                   CASE WHEN hour(date) < 12 THEN 1 ELSE 0 END AS is_am,
                   open, high, low, close,
                   COALESCE(ln(NULLIF(close, 0) / NULLIF(close_p, 0)), 0)   AS r,
                   GREATEST(volume - COALESCE(volume_p, 0), 0)              AS v,
                   GREATEST(amount - COALESCE(amount_p, 0), 0)              AS a,
                   GREATEST(deal_number - COALESCE(deal_p, 0), 0)           AS tn,
                   CASE WHEN bp1 > 0 AND ap1 > bp1 THEN 1 ELSE 0 END        AS bok,
                   (ap1 + bp1) / 2.0                                        AS mid,
                   ap1, bp1, ap5, bp5, av1, bv1, bidv5, askv5, bidn5, askn5,
                   bp1_p, ap1_p, bv1_p, av1_p, bidv5_p, askv5_p
            FROM w
        ),
        e AS (  -- 每分钟派生量;无效盘口(bok=0)置 NULL,聚合自动忽略
            SELECT instrument, d, rn, rn_desc, nb, is_am, open, high, low, close, r, v, a, tn,
                CASE WHEN bok = 1 THEN (bidv5 - askv5) / NULLIF(bidv5 + askv5, 0) END AS dimb,
                CASE WHEN bok = 1 THEN (bv1 - av1) / NULLIF(bv1 + av1, 0) END         AS dimb1,
                CASE WHEN bok = 1 THEN (bidn5 - askn5) / NULLIF(bidn5 + askn5, 0) END AS oimb,
                CASE WHEN bok = 1 AND bidv5 > 0 AND askv5 > 0 AND bidn5 > 0 AND askn5 > 0
                     THEN ln((bidv5 / bidn5) / NULLIF(askv5 / askn5, 0)) END          AS ordsz,
                CASE WHEN bok = 1 THEN (ap1 - bp1) / NULLIF(mid, 0) END               AS spr,
                CASE WHEN bok = 1 AND bv1 + av1 > 0
                     THEN (ap1*bv1 + bp1*av1) / NULLIF(bv1 + av1, 0) / NULLIF(mid, 0) - 1 END AS mpd,
                CASE WHEN bok = 1 AND bp5 > 0 AND ap5 > 0
                     THEN (ap5 - ap1) / NULLIF(mid, 0) - (bp1 - bp5) / NULLIF(mid, 0) END     AS slope_asym,
                CASE WHEN bok = 1 THEN bv1 + av1 END                                  AS depth1,
                CASE WHEN bok = 1 THEN bidv5 + askv5 END                              AS depth5,
                CASE WHEN bok = 1 AND bp1_p > 0 AND ap1_p > bp1_p THEN
                      (CASE WHEN bp1 >= bp1_p THEN bv1   ELSE 0 END)
                    - (CASE WHEN bp1 <= bp1_p THEN bv1_p ELSE 0 END)
                    - (CASE WHEN ap1 <= ap1_p THEN av1   ELSE 0 END)
                    + (CASE WHEN ap1 >= ap1_p THEN av1_p ELSE 0 END) END              AS ofi1,
                CASE WHEN bok = 1 AND bidv5_p IS NOT NULL AND bp1_p > 0
                     THEN (bidv5 - bidv5_p) - (askv5 - askv5_p) END                   AS ofi5,
                CASE WHEN bp1_p IS NOT NULL AND (bp1 <> bp1_p OR ap1 <> ap1_p)
                     THEN 1 ELSE 0 END                                                AS qchg
            FROM m
        )
        SELECT instrument, d,
            MAX(nb) AS nb,
            SUM(CASE WHEN dimb IS NOT NULL THEN 1 ELSE 0 END)          AS n_book,
            MAX(CASE WHEN rn = 1 THEN open END)                        AS open_first,
            MAX(CASE WHEN rn = nb THEN close END)                      AS close_last,
            MAX(high) AS hi, MIN(low) AS lo,
            SUM(r) AS s_r, SUM(r*r) AS s_r2, SUM(r*r*r) AS s_r3, SUM(r*r*r*r) AS s_r4,
            SUM(ABS(r)) AS s_absr,
            SUM(CASE WHEN r > 0 THEN 1 ELSE 0 END)                     AS n_up,
            SUM(CASE WHEN r <> 0 THEN 1 ELSE 0 END)                    AS n_nz,
            SUM(v) AS s_v, SUM(v*v) AS s_v2, SUM(a) AS s_a, SUM(tn) AS s_tn,
            SUM(r*v) AS s_rv, SUM(ABS(r)*v) AS s_absrv,
            SUM(close*v) AS s_cv,
            SUM(CASE WHEN rn_desc <= 30 THEN close*v ELSE 0 END)       AS cv_last30,
            SUM(CASE WHEN rn <= 30 THEN v ELSE 0 END)                  AS v_first30,
            SUM(CASE WHEN rn_desc <= 30 THEN v ELSE 0 END)             AS v_last30,
            SUM(CASE WHEN rn <= 30 THEN r ELSE 0 END)                  AS r_first30,
            SUM(CASE WHEN rn_desc <= 30 THEN r ELSE 0 END)             AS r_last30,
            SUM(CASE WHEN is_am = 1 THEN r ELSE 0 END)                 AS r_am,
            SUM(CASE WHEN is_am = 0 THEN r ELSE 0 END)                 AS r_pm,
            SUM(ofi1) AS ofi1_sum, SUM(ofi5) AS ofi5_sum,
            AVG(depth1) AS depth1_avg, AVG(depth5) AS depth5_avg,
            SUM(dimb) AS s_dimb, SUM(dimb*dimb) AS s_dimb2,
            AVG(CASE WHEN rn_desc <= 30 THEN dimb END)                 AS dimb_last30,
            AVG(dimb1) AS dimb1_avg, AVG(oimb) AS oimb_avg, AVG(ordsz) AS ordsz_avg,
            SUM(spr) AS s_spr, SUM(spr*spr) AS s_spr2,
            SUM(CASE WHEN spr IS NOT NULL THEN 1 ELSE 0 END)           AS n_spr,
            AVG(CASE WHEN rn_desc <= 30 THEN spr END)                  AS spr_last30,
            AVG(mpd) AS mpd_avg, AVG(slope_asym) AS slope_asym_avg,
            SUM(qchg) AS qchg_sum
        FROM e
        GROUP BY instrument, d
        """


    def finish_primitives(g):
        """充分统计量 → 32 个日频原语(全 pandas 向量化,含全部除零/退化保护)。"""
        eps = 1e-12
        n = g["nb"].astype("float64")
        nbk = g["n_book"].astype("float64")

        def safe_div(a, b):
            b = np.asarray(b, dtype="float64")
            return np.where(np.abs(b) > eps, np.asarray(a, dtype="float64") / np.where(np.abs(b) > eps, b, 1.0), np.nan)

        out = pd.DataFrame({
            "date": pd.to_datetime(g["d"]),
            "instrument": g["instrument"].astype(str),
        })

        # ---- 分钟收益矩 ----
        mu = g["s_r"] / n
        m2 = (g["s_r2"] / n - mu ** 2).clip(lower=0.0)
        sd = np.sqrt(m2)
        m3 = g["s_r3"] / n - 3 * mu * g["s_r2"] / n + 2 * mu ** 3
        m4 = g["s_r4"] / n - 4 * mu * g["s_r3"] / n + 6 * mu ** 2 * g["s_r2"] / n - 3 * mu ** 4

        ok_px = (g["open_first"] > 0) & (g["close_last"] > 0)
        ret_oc = pd.Series(np.where(ok_px, np.log(g["close_last"].where(ok_px, 1.0) / g["open_first"].where(ok_px, 1.0)), np.nan), index=g.index)

        # ---- 价格路径 ----
        out["pv_ret_oc"] = ret_oc
        out["pv_ret_am"] = g["r_am"]
        out["pv_ret_pm"] = g["r_pm"]
        out["pv_ret_first30"] = g["r_first30"]
        out["pv_ret_last30"] = g["r_last30"]
        rng = g["hi"] - g["lo"]
        out["pv_pos_close"] = np.where(rng > eps, (g["close_last"] - g["lo"]) / np.where(rng > eps, rng, 1.0), 0.5)
        vwap = safe_div(g["s_cv"], g["s_v"])
        out["pv_vwap_dev"] = safe_div(g["close_last"], vwap) - 1.0
        vwap_l30 = safe_div(g["cv_last30"], g["v_last30"])
        out["pv_vwap_dev_last30"] = safe_div(vwap_l30, vwap) - 1.0
        out["pv_range"] = safe_div(rng, vwap)

        # ---- 分钟收益分布 ----
        out["rd_std"] = sd
        out["rd_skew"] = np.where(sd > eps, m3 / np.where(sd > eps, sd ** 3, 1.0), np.nan)
        out["rd_kurt"] = np.where(m2 > eps, m4 / np.where(m2 > eps, m2 ** 2, 1.0) - 3.0, np.nan)
        out["rd_upshare"] = np.where(g["n_nz"] > 0, g["n_up"] / np.where(g["n_nz"] > 0, g["n_nz"], 1.0), 0.5)
        out["rd_trend_eff"] = np.abs(ret_oc) / (g["s_absr"] + eps)

        # ---- 量能时序 ----
        out["vt_hhi"] = n * safe_div(g["s_v2"], g["s_v"] ** 2)
        out["vt_first30_share"] = safe_div(g["v_first30"], g["s_v"])
        out["vt_last30_share"] = safe_div(g["v_last30"], g["s_v"])
        mv = g["s_v"] / n
        sdv = np.sqrt((g["s_v2"] / n - mv ** 2).clip(lower=0.0))
        cov_rv = g["s_rv"] / n - mu * mv
        out["vt_ret_vol_corr"] = np.where((sd > eps) & (sdv > eps), cov_rv / (sd * sdv + eps), np.nan)
        mabs = g["s_absr"] / n
        sd_abs = np.sqrt((g["s_r2"] / n - mabs ** 2).clip(lower=0.0))
        cov_av = g["s_absrv"] / n - mabs * mv
        out["vt_absret_vol_corr"] = np.where((sd_abs > eps) & (sdv > eps), cov_av / (sd_abs * sdv + eps), np.nan)
        out["vt_avg_trade_size"] = np.where((g["s_v"] > 0) & (g["s_tn"] > 0),
                                            np.log(np.where(g["s_tn"] > 0, g["s_v"] / np.where(g["s_tn"] > 0, g["s_tn"], 1.0), 1.0)), np.nan)
        out["vt_amihud"] = np.abs(ret_oc) / (g["s_a"] / 1e8 + eps)
        out["vt_vw_ret"] = safe_div(g["s_rv"], g["s_v"])

        # ---- 五档盘口 ----
        out["ob_ofi1"] = safe_div(g["ofi1_sum"], nbk * g["depth1_avg"])
        out["ob_ofi5"] = safe_div(g["ofi5_sum"], nbk * g["depth5_avg"])
        dimb_avg = safe_div(g["s_dimb"], nbk)
        out["ob_dimb_mean"] = dimb_avg
        out["ob_dimb_std"] = np.sqrt(np.clip(safe_div(g["s_dimb2"], nbk) - dimb_avg ** 2, 0.0, None))
        out["ob_dimb_last30"] = g["dimb_last30"]
        out["ob_dimb1_mean"] = g["dimb1_avg"]
        out["ob_oimb_mean"] = g["oimb_avg"]
        out["ob_ordsize_imb"] = g["ordsz_avg"]
        spr_avg = safe_div(g["s_spr"], g["n_spr"])
        out["ob_spread_mean"] = spr_avg
        out["ob_spread_std"] = np.sqrt(np.clip(safe_div(g["s_spr2"], g["n_spr"]) - spr_avg ** 2, 0.0, None))
        out["ob_spread_last30"] = g["spr_last30"]
        out["ob_mpd_mean"] = g["mpd_avg"]
        out["ob_slope_asym"] = g["slope_asym_avg"]
        out["ob_quote_int"] = safe_div(g["qchg_sum"], n)
        out["ob_book_to_flow"] = np.where((g["depth5_avg"] > 0),
                                          np.log(np.where(g["depth5_avg"] > 0, g["depth5_avg"], 1.0) / (g["s_v"] / n + 1.0)), np.nan)

        for c in PRIM_COLS:
            out[c] = pd.to_numeric(out[c], errors="coerce").astype("float32")
            out[c] = out[c].replace([np.inf, -np.inf], np.nan)
        return out[["date", "instrument"] + PRIM_COLS]


    def build_primitives(q, bar_table, inst_table, d0, d1, min_bars=MIN_BARS):
        """按月分块拉充分统计量 → finish → 拼接全期原语宽表。q(sql, filters) -> DataFrame。"""
        try:
            import psutil

            def _avail():
                return f"{psutil.virtual_memory().available / 2**30:.1f}GB"
        except Exception:
            def _avail():
                return "n/a"

        chunks = month_chunks(d0, d1)
        parts = []
        for i, (cs, ce) in enumerate(chunks):
            stats = q(primitive_sql(bar_table, inst_table, cs, ce),
                      {"date": [cs, ce + " 23:59:59"]})
            if len(stats) == 0:
                print(f"[f006-step1] 分块 {i+1}/{len(chunks)} ({cs}..{ce}): 0 行,跳过")
                continue
            # DAI 的 SUM(整数列) 返回 HUGEINT/DECIMAL → .df() 是 object 列(Decimal),
            # 直接参与算术会 TypeError(2026-07-13 平台实测;本地 DuckDB 自动转 float 故冒烟不显)
            for c in stats.columns:
                if c not in ("instrument", "d"):
                    stats[c] = pd.to_numeric(stats[c], errors="coerce").astype("float64")
            stats = stats[stats["nb"] >= min_bars]
            parts.append(finish_primitives(stats))
            del stats
            gc.collect()
            print(f"[f006-step1] 分块 {i+1}/{len(chunks)} ({cs}..{ce}): "
                  f"{len(parts[-1])} stock-day | 剩余内存 {_avail()}")
        if not parts:
            raise ValueError(f"build_primitives: [{d0},{d1}] 无数据")
        prim = pd.concat(parts, ignore_index=True)
        prim = prim.sort_values(["instrument", "date"]).reset_index(drop=True)
        return prim


    # ==================== 模型(与训练侧 pana_model.py 原文同构) ====================
    def _encoder(d, n_layers, n_heads, drop):
        layer = nn.TransformerEncoderLayer(
            d_model=d, nhead=n_heads, dim_feedforward=4 * d, dropout=drop,
            batch_first=True, norm_first=True, activation="gelu",
        )
        return nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)

    class PANA(nn.Module):
        def __init__(self, n_feat, d=64, T=20, k_heads=6, n_temporal=2, n_cs=1,
                     n_heads_attn=4, drop=0.0, input_drop=0.1):
            # drop 默认 0:trunk 内部 dropout 会给各头叠加独立噪声,把训练中测得的头间相关假性
            # 压低、骗过去相关损失(run001 教训:训练 div≈0.005 ↔ eval 实测 0.73)。正则改用
            # 输入层 dropout(共享噪声,不扭曲头间相关测量)+ 权重衰减。
            super().__init__()
            self.T, self.k = T, k_heads
            self.in_drop = nn.Dropout(input_drop)
            self.proj = nn.Linear(n_feat, d)
            self.pos = nn.Parameter(torch.zeros(1, T, d))
            self.temporal = _encoder(d, n_temporal, n_heads_attn, drop)
            self.mkt_token = nn.Parameter(torch.zeros(1, 1, d))
            self.cs = _encoder(d, n_cs, n_heads_attn, drop)
            self.norm = nn.LayerNorm(d)
            self.heads = nn.Linear(d, k_heads)
            nn.init.trunc_normal_(self.pos, std=0.02)
            nn.init.trunc_normal_(self.mkt_token, std=0.02)

        def forward(self, x):
            """x: (n, T, F) 当日有效股票的 T 日原语窗 → z: (n, K) 因子头原值(未标准化)。"""
            h = self.temporal(self.proj(self.in_drop(x)) + self.pos)  # (n, T, d)
            h = h[:, -1, :]                                       # 取末 token = 当日状态
            tok = torch.cat([self.mkt_token, h.unsqueeze(0)], 1)  # (1, n+1, d) 市场token在首位
            h = self.cs(tok)[0, 1:, :]                            # 截面互注意,去掉市场token
            return self.heads(self.norm(h))                       # (n, K)


    # ==================== 推理核心(与本地平价校验同一份代码) ====================
    def panel_from_prim(prim, prim_cols, cov_min=0.5):
        """原语长表 → 面板张量(与训练侧同约定)。
        prim: DataFrame[date, instrument, *prim_cols] → (dates, insts, X(D,N,F) float32, Vf(D,N) bool)
        每特征逐日截面百分秩((0,1],并列取均值) → (r-0.5)*sqrt(12);缺失→0;覆盖率<cov_min → 无效。"""
        dates = np.sort(prim["date"].unique())
        insts = np.sort(prim["instrument"].unique())
        D, N, F = len(dates), len(insts), len(prim_cols)
        X = np.zeros((D, N, F), np.float32)
        cov = np.zeros((D, N), np.int16)
        for i, c in enumerate(prim_cols):
            piv = prim.pivot_table(index="date", columns="instrument", values=c, aggfunc="first") \
                .reindex(index=dates, columns=insts)
            m = piv.to_numpy(np.float32)
            cov += (~np.isnan(m)).astype(np.int16)
            r = pd.DataFrame(m).rank(axis=1, pct=True).to_numpy(np.float32)
            X[:, :, i] = np.nan_to_num((r - 0.5) * _SQRT12, nan=0.0)
        Vf = cov >= int(np.ceil(cov_min * F))
        return dates, insts, X, Vf

    def infer_factor(model, X, Vf, T, head, device="cpu", min_xsec=30):
        """逐日推理:d 日取 [d-T+1, d] 窗、当日有效股票 → 头 head 输出按日截面 zscore。
        返回 (D,N) float32,无效位 NaN。要求调用方保证 d>=T-1 的日子才用(前缓冲由查询窗保证)。"""
        D, N, _ = X.shape
        out = np.full((D, N), np.nan, np.float32)
        Xt = torch.from_numpy(X).to(device)
        Vt = torch.from_numpy(Vf).to(device)
        model.eval()
        with torch.no_grad():
            for d in range(T - 1, D):
                idx = Vt[d].nonzero(as_tuple=True)[0]
                if len(idx) < min_xsec:
                    continue
                xw = Xt[d - T + 1:d + 1, idx].permute(1, 0, 2)
                z = model(xw)[:, head].float()
                z = (z - z.mean()) / (z.std() + 1e-8)
                out[d, idx.cpu().numpy()] = z.cpu().numpy()
        return out


    # ==================== 权重与推理 ====================
    def find_weights(name):
        cands = [name, os.path.join(os.getcwd(), name),
                 os.path.join(os.path.dirname(os.getcwd()), name),
                 os.path.join(os.path.expanduser("~"), "work", name),
                 os.path.join("/home/aiuser/work", name)]
        for p in cands:
            if os.path.isfile(p):
                return p
        raise FileNotFoundError(f"未找到权重 {name},请与 notebook 一起上传")

    with open(find_weights("f019_weights.json"), "r") as f:
        payload = json.load(f)
    if payload["features"] != PRIM_COLS:
        raise ValueError("权重 features 与代码 PRIM_COLS 不一致,须用同版本重新导出")
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    ar = payload["arch"]
    model = PANA(n_feat=ar["n_feat"], d=ar["d"], T=ar["T"], k_heads=ar["k_heads"],
                 n_temporal=ar["n_temporal"], n_cs=ar["n_cs"],
                 n_heads_attn=ar["n_heads_attn"], drop=0.0, input_drop=0.0).to(dev)
    model.load_state_dict({k: torch.tensor(np.array(v, dtype=np.float32))
                           for k, v in payload["state_dict"].items()})
    print(f"[f012-infer] 权重 {len(payload['state_dict'])} 张量 | head={HEAD} | device={dev}")

    def q(sql, filters):
        return dai.query(sql, filters=filters, compression=True).df()

    sd, ed = str(start_date)[:10], str(end_date)[:10]
    q_start = (pd.Timestamp(sd) - pd.Timedelta(days=50)).strftime("%Y-%m-%d")  # T=20 交易日预热
    bar1m = datasources["bar1m"]
    prim = build_primitives(q, bar1m, "bigalpha_2026_instruments", q_start, ed)
    prim["instrument"] = prim["instrument"].astype(str)
    gc.collect()

    dates_, insts_, Xp, Vfp = panel_from_prim(prim, PRIM_COLS)
    fac = infer_factor(model, Xp, Vfp, T, HEAD, device=dev)
    df = pd.DataFrame(fac, index=pd.to_datetime(dates_), columns=insts_)         .stack().rename("factor").reset_index()
    df.columns = ["date", "instrument", "factor"]
    df = df[df["date"].between(start_date, end_date)]

    # 危机日护甲:中证1000 成分全集左连 + 当日截面中位数填充(f009 教训,覆盖恒 100%)
    pool = q("SELECT date, instrument FROM bigalpha_2026_instruments", {"date": [sd, ed]})
    pool["date"] = pd.to_datetime(pool["date"])
    pool["instrument"] = pool["instrument"].astype(str)
    pool = pool[pool["date"].between(start_date, end_date)][["date", "instrument"]]
    out = pd.merge(pool, df, how="left", on=["date", "instrument"])
    med = out.groupby("date")["factor"].transform("median")
    out["factor"] = out["factor"].fillna(med).fillna(0.0)
    out["factor"] = out["factor"].replace([np.inf, -np.inf], 0.0)
    return out[["date", "instrument", "factor"]]
